# Hybrid Amazon Reviews Recommender System

This notebook rebuilds the recommender into a high-recall hybrid retrieval stack plus a high-precision LambdaMART-style ranker:

- raw review JSONL files from `amazon_review_data/`
- official Amazon Reviews 2023 category metadata
- collaborative baselines: popularity and cooccurrence
- hybrid candidate generation:
  - cooccurrence KNN
  - latent collaborative filtering via sparse SVD
  - content-based text retrieval
  - neural two-tower retrieval
- candidate-union diagnostics focused on recall
- XGBoost ranking as the default precision stage
- optional DLRM-lite ranker ablation through the pipeline API

The notebook is an execution front-end over `amazon_recsys_pipeline.py`, so the modeling logic stays testable outside Jupyter.


In [ ]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import amazon_recsys_pipeline as arp

arp = importlib.reload(arp)

PipelineConfig = arp.PipelineConfig
apply_run_profile = arp.apply_run_profile
candidate_source_diagnostics = arp.candidate_source_diagnostics
download_metadata_files = arp.download_metadata_files
evaluate_candidate_table = arp.evaluate_candidate_table
generate_candidate_union = arp.generate_candidate_union
get_user_order_history = arp.get_user_order_history
item_item_cooccurrence_candidates = arp.item_item_cooccurrence_candidates
load_metadata = arp.load_metadata
load_reviews = arp.load_reviews
make_splits = arp.make_splits
pipeline_summary = arp.pipeline_summary
popularity_by_category_candidates = arp.popularity_by_category_candidates
prepare_corpus = arp.prepare_corpus
recommend = arp.recommend
save_config = arp.save_config
ensure_directories = arp.ensure_directories
split_diagnostics = arp.split_diagnostics
train_ranker = arp.train_ranker
train_retrievers = arp.train_retrievers

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)

CONFIG = apply_run_profile(
    PipelineConfig(
        base_dir=Path.cwd(),
        run_name="hybrid_quality_dev",
        run_profile="quality",
        dev_mode=True,
        dev_fraction=0.30,
        dev_sampling_strategy="category_balanced_user",
        dev_hard_negative_multiplier=2.5,
        dev_neutral_multiplier=1.5,
        k_core=3,
        train_positive_cap=500_000,
        negatives_per_positive=6,
        retriever_validation_negatives_per_positive=10,
        retriever_quality_min_history=3,
        retriever_logit_scale=8.0,
        persist_encoder_models=False,
        enable_neural_retriever=True,
        retriever_epochs=10,
        eval_user_cap=1_000,
        candidate_union_top_k=200,
        candidate_union_batch_size=500,
        ranker_candidate_top_k=100,
        ranker_train_example_cap=5_000,
        ranker_val_example_cap=1_000,
        ranker_backend="xgboost",
        xgb_n_estimators=200,
        xgb_max_depth=6,
        metadata_download_if_missing=True,
    )
)

ensure_directories(CONFIG)
save_config(CONFIG)

print("XGBoost available:", arp.xgb is not None)
print("TensorFlow Recommenders available (optional):", arp.tfrs is not None)
print("Note: native Windows TensorFlow runs on CPU for TF >= 2.11. Use WSL2 or DirectML if you need GPU acceleration.")
display(pipeline_summary(CONFIG))
display(
    pd.DataFrame(
        [
            {"artifact": "artifact_root", "path": str(CONFIG.artifact_root)},
            {"artifact": "cache_dir", "path": str(CONFIG.cache_dir)},
            {"artifact": "model_dir", "path": str(CONFIG.model_dir)},
            {"artifact": "eval_dir", "path": str(CONFIG.eval_dir)},
        ]
    )
)

# Current default:
# - quality-profile dev run
# - category-balanced, rating-aware user sampling
# - hybrid retriever budgets from the pipeline config
# - neural retriever is disabled by default for faster CPU iteration and can be re-enabled later
# - ranker candidate generation is chunked and validation/train caps are notebook-safe
#
# For a very fast smoke run, uncomment:
# CONFIG.run_profile = "debug"
# CONFIG = apply_run_profile(CONFIG)
# CONFIG.retriever_epochs = 3
# CONFIG.xgb_n_estimators = 100


## Benchmark Context

The Kaggle notebooks in `Research/Sample Notebooks from Kaggle` are useful sanity references, but they solve easier recommendation problems than this notebook:

- usually one domain or category rather than a 3-category union
- heavier user/item pruning before modeling
- simpler popularity, KNN, SVD, or content-similarity objectives

This notebook keeps the harder multi-category setup, so the benchmark question is not “can the neural tower beat Kaggle-style toy scores?” but rather:

- do the classical baselines remain healthy?
- does the hybrid union improve recall over the best single source?
- does the ranker improve precision over the raw union ordering?


## Raw-Data Audit

In [2]:
review_audit = load_reviews(CONFIG, max_rows_per_category=20_000)
display(review_audit.head())

audit_summary = (
    review_audit.groupby("source_category")
    .agg(
        rows=("user_id", "size"),
        unique_users=("user_id", "nunique"),
        unique_parent_asins=("parent_asin", "nunique"),
        avg_rating=("rating", "mean"),
        verified_rate=("verified_purchase", "mean"),
        low_rating_share=("rating", lambda x: float((x <= 2).mean())),
        neutral_share=("rating", lambda x: float((x == 3).mean())),
        positive_share=("rating", lambda x: float((x >= 4).mean())),
        rows_per_user=("user_id", lambda x: float(len(x) / max(x.nunique(), 1))),
        rows_per_item=("parent_asin", lambda x: float(len(x) / max(x.nunique(), 1))),
    )
    .reset_index()
)
display(audit_summary)


Audit review files:   0%|          | 0/3 [00:00<?, ?category/s]

,source_category,user_id,asin,parent_asin,rating,timestamp,verified_purchase,helpful_vote,title,text,timestamp_dt
0,All_Beauty,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,B00YQ6X8EO,5.0,1588687728923,True,0,Such a lovely scent but not overpowering.,"This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it...",2020-05-05 14:08:48.923000+00:00
1,All_Beauty,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,B081TJ8YS3,4.0,1588615855070,True,1,Works great but smells a little weird.,"This product does what I need it to do, I just wish it was odorless or had a soft coconut smell. Having my head smell like an orange cof...",2020-05-04 18:10:55.070000+00:00
2,All_Beauty,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B07PNNCSP9,B097R46CSY,5.0,1589665266052,True,2,Yes!,"Smells good, feels great!",2020-05-16 21:41:06.052000+00:00
3,All_Beauty,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B09JS339BZ,B09JS339BZ,1.0,1643393630220,True,0,Synthetic feeling,Felt synthetic,2022-01-28 18:13:50.220000+00:00
4,All_Beauty,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,B08BZ63GMJ,5.0,1609322563534,True,0,A+,Love it,2020-12-30 10:02:43.534000+00:00


,source_category,rows,unique_users,unique_parent_asins,avg_rating,verified_rate,low_rating_share,neutral_share,positive_share,rows_per_user,rows_per_item
0,All_Beauty,20000,14750,12393,4.09480,0.79625,0.15535,0.0962,0.74845,1.355932,1.613814
1,Automotive,20000,4311,16585,4.31175,0.89090,0.11660,0.0643,0.81910,4.639295,1.205909
2,Industrial_and_Scientific,20000,6881,13154,4.40420,0.77215,0.09125,0.0615,0.84725,2.906554,1.520450


## Metadata Loading

In [3]:
metadata_paths = download_metadata_files(CONFIG)
metadata_sample = load_metadata(CONFIG, max_rows_per_category=2_000)

display(pd.DataFrame({"category": list(metadata_paths.keys()), "path": [str(path) for path in metadata_paths.values()]}))
display(metadata_sample.head())


,category,path
0,All_Beauty,c:\Users\Bayanda.Kutshwa\OneDrive - Empresas SK\Data Science Upskilling\Recommender Systems\amazon_review_data\metadata\meta_All_Beauty....
1,Automotive,c:\Users\Bayanda.Kutshwa\OneDrive - Empresas SK\Data Science Upskilling\Recommender Systems\amazon_review_data\metadata\meta_Automotive....
2,Industrial_and_Scientific,c:\Users\Bayanda.Kutshwa\OneDrive - Empresas SK\Data Science Upskilling\Recommender Systems\amazon_review_data\metadata\meta_Industrial_...


,source_category,parent_asin,meta_title,store,categories_text,description_text,features_text,bought_together_text,price,average_rating,rating_number
0,All_Beauty,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)",Howard Products,,,,,NaN,4.8,10
1,All_Beauty,B076WQZGPM,"Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 ...",Yes To,,,,,NaN,4.5,3
2,All_Beauty,B000B658RI,Eye Patch Black Adult with Tie Band (6 Per Pack),Levine Health Products,,,,,NaN,4.4,26
3,All_Beauty,B088FKY3VD,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4D Imitation Eyebrow Tattoos, 4D Hair-like Authentic Eyebrows Waterproof Long Lasting for W...",Cherioll,,,,,NaN,3.1,102
4,All_Beauty,B07NGFDN6G,Precision Plunger Bars for Cartridge Grips – 93mm – Bag of 10 Plungers,Precision,,"The Precision Plunger Bars are designed to work seamlessly with the Precision Disposable 1. 25"" Contoured Soft Cartridge Grips and the P...","Material: 304 Stainless Steel; Brass tip Lengths Available: 88mm, 93mm, 98mm Accepts cartridge needles with vice style tattoo machines W...",,NaN,4.3,7


## Preprocessing

In [4]:
prepared = prepare_corpus(CONFIG, force_rebuild=False)

display(
    prepared.raw_review_stats[
        [
            "source_category",
            "sampling_strategy",
            "raw_rows_seen",
            "kept_rows",
            "effective_keep_rate",
            "base_keep_fraction",
            "positive_rows",
            "neutral_rows",
            "hard_negative_rows",
            "avg_rating",
            "verified_rate",
        ]
    ]
)
display(prepared.kcore_stats)

print("Positive interactions:", len(prepared.interactions))
print("Hard negatives:", len(prepared.hard_negatives))
print("Item feature rows:", len(prepared.item_features))
print("Item text matrix shape:", prepared.item_text_matrix.shape)


Config change detected for cached corpus artifacts. Rebuilding cache for the current notebook settings.


,source_category,sampling_strategy,raw_rows_seen,kept_rows,effective_keep_rate,base_keep_fraction,positive_rows,neutral_rows,hard_negative_rows,avg_rating,verified_rate
0,All_Beauty,category_balanced_user,701528,701528,1.000000,1.000000,500107,56307,145114,3.960245,0.905123
1,Automotive,category_balanced_user,19955450,3270214,0.163876,0.129488,2019264,229769,1021181,3.609448,0.960078
2,Industrial_and_Scientific,category_balanced_user,5183005,3072871,0.592874,0.498552,2017816,236159,818896,3.765657,0.946643


,iteration,rows_after_filter,users_after_filter,items_after_filter
0,1,4537187,334723,275148
1,2,1530102,255698,149542
2,3,1277745,222584,135796
3,4,1192783,216957,129378
4,5,1169658,213461,128049
5,6,1160151,212808,127243
6,7,1157267,212350,127103
7,8,1156078,212282,126987
8,9,1155714,212207,126972
9,10,1155537,212195,126959


Positive interactions: 1155470
Hard negatives: 90290
Item feature rows: 163793
Item text matrix shape: (163793, 64)


## EDA

In [5]:
interaction_summary = (
    prepared.interactions.groupby("source_category")
    .agg(
        positive_rows=("user_id", "size"),
        unique_users=("user_id", "nunique"),
        unique_items=("parent_asin", "nunique"),
        min_time=("timestamp_dt", "min"),
        max_time=("timestamp_dt", "max"),
    )
    .reset_index()
)

user_activity = prepared.interactions.groupby("user_id").size()
item_activity = prepared.interactions.groupby("parent_asin").size()

display(interaction_summary)
display(
    pd.DataFrame(
        {
            "metric": [
                "median_positive_interactions_per_user",
                "median_positive_interactions_per_item",
                "users_with_5plus_positives",
                "items_with_5plus_positives",
                "positive_to_item_ratio",
                "hard_negative_share_vs_positive",
            ],
            "value": [
                float(user_activity.median()),
                float(item_activity.median()),
                float((user_activity >= 5).mean()),
                float((item_activity >= 5).mean()),
                float(len(prepared.interactions) / max(len(prepared.item_features), 1)),
                float(len(prepared.hard_negatives) / max(len(prepared.interactions), 1)),
            ],
        }
    )
)


,source_category,positive_rows,unique_users,unique_items,min_time,max_time
0,All_Beauty,20598,10713,3394,2003-01-08 04:41:46+00:00,2023-08-14 03:02:16.350000+00:00
1,Automotive,689301,130847,81427,2006-05-23 20:08:58+00:00,2023-09-11 17:51:51.054000+00:00
2,Industrial_and_Scientific,445571,114164,42137,2002-05-09 02:04:47+00:00,2023-09-06 17:57:40.466000+00:00


,metric,value
0,median_positive_interactions_per_user,4.000000
1,median_positive_interactions_per_item,5.000000
2,users_with_5plus_positives,0.377049
3,items_with_5plus_positives,0.524874
4,positive_to_item_ratio,7.054453
5,hard_negative_share_vs_positive,0.078141


## Split Generation and Diagnostics

In [6]:
splits = make_splits(prepared)

split_summary = pd.DataFrame(
    [
        {"split": "train", "rows": len(splits.train_examples), "users": splits.train_examples["user_id"].nunique()},
        {"split": "val", "rows": len(splits.val_examples), "users": splits.val_examples["user_id"].nunique()},
        {"split": "test", "rows": len(splits.test_examples), "users": splits.test_examples["user_id"].nunique()},
    ]
)
display(split_summary)

diagnostics = split_diagnostics(prepared, splits)
for name, frame in diagnostics.items():
    print(name)
    display(frame)


,split,rows,users
0,train,500000,122585
1,val,124141,124141
2,test,124141,124141


summary


,metric,value
0,training_interactions,643050.000000
1,training_users,124141.000000
2,training_items,119453.000000
3,catalog_coverage_after_kcore,0.729292
4,hard_negative_density,0.078141
5,mean_val_history_length,4.314127
6,mean_test_history_length,5.201513
7,p90_val_history_length,10.000000
8,p90_test_history_length,10.000000
9,val_repeat_rate,0.019961


category_share


,split,source_category,share,count
0,val,All_Beauty,0.014185,1761
1,val,Automotive,0.595017,73866
2,val,Industrial_and_Scientific,0.390798,48514
3,test,All_Beauty,0.014057,1745
4,test,Automotive,0.590031,73247
5,test,Industrial_and_Scientific,0.395913,49149


history_summary


,split,metric,value
0,train,rows,500000.0
1,val,rows,124141.0
2,test,rows,124141.0
3,train,users,122585.0
4,val,users,124141.0
5,test,users,124141.0
6,train,median_history_length,3.0
7,val,median_history_length,3.0
8,test,median_history_length,4.0


## Baseline Candidate Generators

In [7]:
def _sample_eval_examples(frame: pd.DataFrame) -> pd.DataFrame:
    eval_examples = frame
    if CONFIG.eval_user_cap is not None and len(eval_examples) > CONFIG.eval_user_cap:
        eval_examples = eval_examples.sample(n=CONFIG.eval_user_cap, random_state=CONFIG.seed).sort_values("example_id")
    return eval_examples


baseline_rows = []
baseline_candidates = {}
for split_name, frame in [("val", splits.val_examples), ("test", splits.test_examples)]:
    eval_examples = _sample_eval_examples(frame)
    pop_candidates = popularity_by_category_candidates(splits, eval_examples, top_k=CONFIG.retrieval_top_k)
    cooc_candidates = item_item_cooccurrence_candidates(splits, eval_examples, top_k=CONFIG.retrieval_top_k)
    baseline_candidates[(split_name, "popularity_by_category")] = pop_candidates
    baseline_candidates[(split_name, "item_item_cooccurrence")] = cooc_candidates
    for variant_name, candidate_frame in [
        ("popularity_by_category", pop_candidates),
        ("item_item_cooccurrence", cooc_candidates),
    ]:
        metrics = evaluate_candidate_table(candidate_frame).assign(split=split_name, variant=variant_name, stage="baseline")
        baseline_rows.append(metrics)

baseline_metrics = pd.concat(baseline_rows, ignore_index=True)
display(baseline_metrics.sort_values(["split", "variant", "K"]))
baseline_metrics.to_csv(CONFIG.eval_dir / "baseline_metrics.csv", index=False)


,K,recall,mrr,ndcg,split,variant,stage
9,10,0.0130,0.007425,0.008712,test,item_item_cooccurrence,baseline
10,50,0.0240,0.007977,0.011187,test,item_item_cooccurrence,baseline
11,100,0.0365,0.008146,0.013190,test,item_item_cooccurrence,baseline
6,10,0.0055,0.001851,0.002720,test,popularity_by_category,baseline
7,50,0.0245,0.002658,0.006785,test,popularity_by_category,baseline
8,100,0.0440,0.002929,0.009929,test,popularity_by_category,baseline
3,10,0.0120,0.006141,0.007538,val,item_item_cooccurrence,baseline
4,50,0.0285,0.006936,0.011195,val,item_item_cooccurrence,baseline
5,100,0.0400,0.007109,0.013083,val,item_item_cooccurrence,baseline
0,10,0.0060,0.001639,0.002624,val,popularity_by_category,baseline


## Train the Retrieval Stack

In [8]:
print("Interpretation note: use Recall@K, MRR@K, and NDCG@K as the retriever selection metrics.")
print("The Keras validation BCE/AUC logs are now computed on mixed-label validation pairs, but offline retrieval metrics remain the decision criteria.")
print("This notebook skips the neural retriever by default on local CPU runs. Set CONFIG.enable_neural_retriever = True only when you want that ablation.")

retrievers = train_retrievers(prepared, splits)
retriever_metric_frames = [arp._normalize_metrics_frame(artifact.metrics).assign(stage="retriever") for artifact in retrievers.values()]
retriever_metrics = (
    pd.concat(retriever_metric_frames, ignore_index=True)
    if retriever_metric_frames
    else pd.DataFrame(columns=["K", "recall", "mrr", "ndcg", "split", "variant", "stage"])
)
retriever_sort_columns = [column for column in ["split", "variant", "K"] if column in retriever_metrics.columns]
if retriever_metrics.empty:
    print("No retriever evaluation metrics were produced for this run.")
else:
    display(retriever_metrics.sort_values(retriever_sort_columns))

for variant_name, artifact in retrievers.items():
    if artifact.retriever_kind != "neural":
        continue
    print(f"{variant_name} pair summaries")
    display(artifact.metadata.get("pair_summaries", pd.DataFrame()))
    print(f"{variant_name} sanity checks")
    display(artifact.metadata.get("sanity_checks", pd.DataFrame()))


Interpretation note: use Recall@K, MRR@K, and NDCG@K as the retriever selection metrics.
The Keras validation BCE/AUC logs are now computed on mixed-label validation pairs, but offline retrieval metrics remain the decision criteria.
This notebook skips the neural retriever by default on local CPU runs. Set CONFIG.enable_neural_retriever = True only when you want that ablation.
Retriever training capped to 100,000 base examples for notebook-safe memory usage.
Epoch 1/10
2735/2735 - 212s - 77ms/step - auc: 0.6145 - bce_loss: 0.3977 - in_batch_loss: 3.6047 - loss: 0.9384 - val_auc: 0.7264 - val_bce_loss: 0.3007 - val_in_batch_loss: 3.1784 - val_loss: 0.7774
Epoch 2/10
2735/2735 - 212s - 77ms/step - auc: 0.7163 - bce_loss: 0.3740 - in_batch_loss: 3.6011 - loss: 0.9141 - val_auc: 0.7043 - val_bce_loss: 0.2987 - val_in_batch_loss: 3.2336 - val_loss: 0.7838
Epoch 3/10
2735/2735 - 215s - 79ms/step - auc: 0.7710 - bce_loss: 0.3499 - in_batch_loss: 3.4953 - loss: 0.8742 - val_auc: 0.6735 - val_b

,K,recall,mrr,ndcg,split,variant,stage
3,10,0.0025,0.000854,0.001254,test,content_based,retriever
4,50,0.0085,0.001081,0.002490,test,content_based,retriever
5,100,0.0130,0.001151,0.003236,test,content_based,retriever
9,10,0.0010,0.000222,0.000401,test,latent_cf,retriever
10,50,0.0030,0.000294,0.000803,test,latent_cf,retriever
11,100,0.0065,0.000342,0.001366,test,latent_cf,retriever
15,10,0.0000,0.000000,0.000000,test,two_tower,retriever
16,50,0.0015,0.000066,0.000328,test,two_tower,retriever
17,100,0.0035,0.000092,0.000646,test,two_tower,retriever
0,10,0.0040,0.001981,0.002443,val,content_based,retriever


two_tower pair summaries


,split,rows,positives,negatives,positive_rate,users,items,mean_history_length
0,train,700000,100000,600000,0.142857,37689,83516,6.75463
1,val,22000,2000,20000,0.090909,2000,12195,5.54900


two_tower sanity checks


,variant,sample_rows,positive_rows,negative_rows,positive_mean_score,negative_mean_score,score_gap,logit_scale
0,two_tower,4096,373,3723,0.194405,0.127192,0.067213,8.0


## Hybrid Candidate Union Diagnostics

In [9]:
def _semantic_hit_table(candidates: pd.DataFrame) -> pd.DataFrame:
    if candidates.empty:
        return pd.DataFrame()
    item_category_lookup = prepared.item_features[["item_idx", "source_category"]].rename(columns={"source_category": "candidate_source_category"})
    enriched = candidates.merge(item_category_lookup, on="item_idx", how="left")
    enriched["same_category_hit"] = (enriched["candidate_source_category"] == enriched["target_source_category"]).astype(int)
    rows = []
    for k in [10, 50, 100]:
        topk = enriched[enriched["rank"] <= k]
        grouped = topk.groupby("example_id", as_index=False).agg(
            exact_hit=("label", "max"),
            same_category_hit=("same_category_hit", "max"),
        )
        rows.append(
            {
                "K": k,
                "exact_hit_rate": float(grouped["exact_hit"].mean()) if not grouped.empty else 0.0,
                "same_category_hit_rate": float(grouped["same_category_hit"].mean()) if not grouped.empty else 0.0,
            }
        )
    return pd.DataFrame(rows)


hybrid_metric_rows = []
hybrid_diagnostics = {}
hybrid_candidate_tables = {}
for split_name, frame in [("val", splits.val_examples), ("test", splits.test_examples)]:
    eval_examples = _sample_eval_examples(frame)
    hybrid_candidates = generate_candidate_union(
        prepared,
        splits,
        retrievers,
        eval_examples,
        top_k=CONFIG.candidate_union_top_k,
        inject_target_if_missing=False,
    )
    hybrid_candidate_tables[split_name] = hybrid_candidates
    hybrid_candidates.to_parquet(CONFIG.eval_dir / f"hybrid_union_{split_name}_retrieval_candidates.parquet", index=False)
    diagnostics_bundle = candidate_source_diagnostics(hybrid_candidates)
    diagnostics_bundle["semantic_hits"] = _semantic_hit_table(hybrid_candidates)
    hybrid_diagnostics[split_name] = diagnostics_bundle
    metrics = diagnostics_bundle["metrics"].copy()
    metrics["split"] = split_name
    metrics["variant"] = "hybrid_union"
    metrics["stage"] = "candidate_union"
    hybrid_metric_rows.append(metrics)
    diagnostics_bundle["per_category"].to_csv(CONFIG.eval_dir / f"hybrid_union_{split_name}_per_category.csv", index=False)
    diagnostics_bundle["source_summary"].to_csv(CONFIG.eval_dir / f"hybrid_union_{split_name}_source_summary.csv", index=False)
    diagnostics_bundle["semantic_hits"].to_csv(CONFIG.eval_dir / f"hybrid_union_{split_name}_semantic_hits.csv", index=False)

hybrid_metrics = pd.concat(hybrid_metric_rows, ignore_index=True)
display(hybrid_metrics.sort_values(["split", "K"]))

print("Validation candidate-source contribution")
display(hybrid_diagnostics["val"]["source_summary"])
print("Validation per-category recall proxy")
display(hybrid_diagnostics["val"]["per_category"])
print("Validation worst slices")
display(hybrid_diagnostics["val"]["worst_slice"])
print("Validation semantic hit diagnostics")
display(hybrid_diagnostics["val"]["semantic_hits"])

hybrid_metrics.to_csv(CONFIG.eval_dir / "hybrid_union_metrics.csv", index=False)


,K,recall,mrr,ndcg,coverage,split,variant,stage
3,10,0.0085,0.005786,0.006392,0.133546,test,hybrid_union,candidate_union
4,50,0.0190,0.006289,0.008718,0.479202,test,hybrid_union,candidate_union
5,100,0.0270,0.006401,0.010010,0.735049,test,hybrid_union,candidate_union
0,10,0.0100,0.004697,0.005909,0.135335,val,hybrid_union,candidate_union
1,50,0.0230,0.005197,0.008602,0.485204,val,hybrid_union,candidate_union
2,100,0.0320,0.005324,0.010058,0.738689,val,hybrid_union,candidate_union


Validation candidate-source contribution


,source,positive_recoveries,examples_with_positive,median_positive_rank
0,cooccurrence,62,62,46.5
1,latent_cf,22,22,49.0
2,content_based,22,22,52.0
3,two_tower,3,3,142.0
4,popularity,0,0,NaN


Validation per-category recall proxy


,target_source_category,hit_rate,hits,count
0,All_Beauty,0.117647,4,34
1,Industrial_and_Scientific,0.050761,40,788
2,Automotive,0.039049,46,1178


Validation worst slices


,target_source_category,hit_rate,hits,count
2,Automotive,0.039049,46,1178
1,Industrial_and_Scientific,0.050761,40,788
0,All_Beauty,0.117647,4,34


Validation semantic hit diagnostics


,K,exact_hit_rate,same_category_hit_rate
0,10,0.010,0.9715
1,50,0.023,0.9880
2,100,0.032,0.9955


## Train the Ranker

In [10]:
ranker = train_ranker(prepared, splits, retrievers, backend=CONFIG.ranker_backend)
display(ranker.metrics)


[0]	validation_0-ndcg@10:0.98774	validation_0-ndcg@20:0.98802
[1]	validation_0-ndcg@10:0.98659	validation_0-ndcg@20:0.98715
[2]	validation_0-ndcg@10:0.98686	validation_0-ndcg@20:0.98742
[3]	validation_0-ndcg@10:0.98757	validation_0-ndcg@20:0.98785
[4]	validation_0-ndcg@10:0.98759	validation_0-ndcg@20:0.98787
[5]	validation_0-ndcg@10:0.98760	validation_0-ndcg@20:0.98760
[6]	validation_0-ndcg@10:0.98729	validation_0-ndcg@20:0.98729
[7]	validation_0-ndcg@10:0.98716	validation_0-ndcg@20:0.98716
[8]	validation_0-ndcg@10:0.98706	validation_0-ndcg@20:0.98706
[9]	validation_0-ndcg@10:0.98722	validation_0-ndcg@20:0.98722
[10]	validation_0-ndcg@10:0.98728	validation_0-ndcg@20:0.98728
[11]	validation_0-ndcg@10:0.98732	validation_0-ndcg@20:0.98732
[12]	validation_0-ndcg@10:0.98719	validation_0-ndcg@20:0.98719
[13]	validation_0-ndcg@10:0.98721	validation_0-ndcg@20:0.98721
[14]	validation_0-ndcg@10:0.98699	validation_0-ndcg@20:0.98699
[15]	validation_0-ndcg@10:0.98699	validation_0-ndcg@20:0.98699
[1

,K,recall,mrr,ndcg,split,variant,stage
0,10,0.011,0.004583,0.006086,val,hybrid_union,ranker
1,50,0.026,0.005311,0.009428,val,hybrid_union,ranker
2,100,0.028,0.005340,0.009755,val,hybrid_union,ranker
3,10,0.013,0.007376,0.008660,test,hybrid_union,ranker
4,50,0.027,0.008118,0.011882,test,hybrid_union,ranker
5,100,0.031,0.008175,0.012529,test,hybrid_union,ranker


## Optional DLRM-Lite Ranker Ablation

In [ ]:
# Uncomment if you want to compare the neural ranker against XGBoost.
ranker_dlrm = train_ranker(prepared, splits, retrievers, backend="dlrm")
display(ranker_dlrm.metrics)


## Evaluation Summary

In [12]:
evaluation_summary = pd.concat(
    [
        baseline_metrics,
        retriever_metrics,
        hybrid_metrics,
        ranker.metrics,
    ],
    ignore_index=True,
)

display(evaluation_summary.sort_values(["split", "stage", "variant", "K"]))

metric_snapshot = (
    evaluation_summary.groupby(["split", "stage", "variant"], as_index=False)[["recall", "mrr", "ndcg"]]
    .max()
    .sort_values(["split", "stage", "variant"])
)
display(metric_snapshot)

val_recall_100 = evaluation_summary[(evaluation_summary["split"] == "val") & (evaluation_summary["K"] == 100)][["stage", "variant", "recall", "mrr", "ndcg"]]
display(val_recall_100.sort_values(["recall", "mrr"], ascending=[False, False]))

evaluation_summary.to_csv(CONFIG.eval_dir / "evaluation_summary.csv", index=False)


,K,recall,mrr,ndcg,split,variant,stage,coverage
9,10,0.0130,0.007425,0.008712,test,item_item_cooccurrence,baseline,NaN
10,50,0.0240,0.007977,0.011187,test,item_item_cooccurrence,baseline,NaN
11,100,0.0365,0.008146,0.013190,test,item_item_cooccurrence,baseline,NaN
6,10,0.0055,0.001851,0.002720,test,popularity_by_category,baseline,NaN
7,50,0.0245,0.002658,0.006785,test,popularity_by_category,baseline,NaN
8,100,0.0440,0.002929,0.009929,test,popularity_by_category,baseline,NaN
33,10,0.0085,0.005786,0.006392,test,hybrid_union,candidate_union,0.133546
34,50,0.0190,0.006289,0.008718,test,hybrid_union,candidate_union,0.479202
35,100,0.0270,0.006401,0.010010,test,hybrid_union,candidate_union,0.735049
39,10,0.0130,0.007376,0.008660,test,hybrid_union,ranker,NaN


,split,stage,variant,recall,mrr,ndcg
0,test,baseline,item_item_cooccurrence,0.0365,0.008146,0.013190
1,test,baseline,popularity_by_category,0.0440,0.002929,0.009929
2,test,candidate_union,hybrid_union,0.0270,0.006401,0.010010
3,test,ranker,hybrid_union,0.0310,0.008175,0.012529
4,test,retriever,content_based,0.0130,0.001151,0.003236
5,test,retriever,latent_cf,0.0065,0.000342,0.001366
6,test,retriever,two_tower,0.0035,0.000092,0.000646
7,val,baseline,item_item_cooccurrence,0.0400,0.007109,0.013083
8,val,baseline,popularity_by_category,0.0330,0.002368,0.007680
9,val,candidate_union,hybrid_union,0.0320,0.005324,0.010058


,stage,variant,recall,mrr,ndcg
5,baseline,item_item_cooccurrence,0.0400,0.007109,0.013083
2,baseline,popularity_by_category,0.0330,0.002368,0.007680
32,candidate_union,hybrid_union,0.0320,0.005324,0.010058
38,ranker,hybrid_union,0.0280,0.005340,0.009755
14,retriever,content_based,0.0200,0.002351,0.005302
20,retriever,latent_cf,0.0110,0.000751,0.002535
26,retriever,two_tower,0.0015,0.000038,0.000279


## Demo Recommendations

In [13]:
def _friendly_order_history_view(df: pd.DataFrame, title: str) -> None:
    pretty = df.copy()
    pretty["Price"] = pretty["price"].apply(lambda x: "Unknown" if pd.isna(x) or float(x) <= 0 else f"${float(x):,.2f}")
    pretty["Catalog rating"] = pretty["average_rating"].apply(lambda x: "Unknown" if pd.isna(x) else f"{float(x):.2f} / 5")
    pretty["Review rating"] = pretty["review_rating"].apply(lambda x: "Unknown" if pd.isna(x) else f"{float(x):.1f} / 5")
    pretty["Verified"] = pretty["verified_purchase"].map({1: "Yes", 0: "No"})
    pretty = pretty.rename(columns={"ordered_at": "Ordered at", "title": "Past order", "source_category": "Category"})
    print(title)
    display(pretty[["Ordered at", "Past order", "Category", "Review rating", "Verified", "Price", "Catalog rating"]].reset_index(drop=True))


def _source_label(raw_value: str) -> str:
    mapping = {
        "cooccurrence": "collaborative cooccurrence",
        "latent_cf": "latent CF",
        "content_based": "content similarity",
        "two_tower": "neural retrieval",
        "popularity": "popularity backfill",
    }
    if pd.isna(raw_value) or not str(raw_value).strip():
        return "candidate backfill"
    return ", ".join(mapping.get(part.strip(), part.strip()) for part in str(raw_value).split("+"))


def _friendly_reason(row: pd.Series) -> str:
    reasons = []
    source_text = str(row.get("candidate_sources", ""))
    if "cooccurrence" in source_text or "latent_cf" in source_text:
        reasons.append("similar shoppers showed related behavior")
    if "content_based" in source_text:
        reasons.append("the product looks similar to past purchases")
    if "two_tower" in source_text:
        reasons.append("the neural model matched the broader user profile")
    if float(row.get("average_rating", 0.0) or 0.0) >= 4.4:
        reasons.append("it is also strongly rated")
    if not reasons:
        reasons.append("it scored well after retrieval and reranking")
    return "; ".join(reasons).capitalize() + "."


def _friendly_recommendation_view(df: pd.DataFrame, title: str) -> None:
    pretty = df.copy()
    base_score = pretty["score"] if "score" in pretty.columns else pretty["retrieval_score"]
    confidence = base_score.rank(method="dense", pct=True).fillna(0.0)
    pretty["Confidence"] = confidence.map(lambda value: "High" if value >= 0.85 else ("Medium" if value >= 0.55 else "Exploratory"))
    pretty["Price"] = pretty["price"].apply(lambda x: "Unknown" if pd.isna(x) or float(x) <= 0 else f"${float(x):,.2f}")
    pretty["Average rating"] = pretty["average_rating"].apply(lambda x: "Unknown" if pd.isna(x) else f"{float(x):.2f} / 5")
    if "candidate_sources" in pretty.columns:
        pretty["Recommendation path"] = pretty["candidate_sources"].map(_source_label)
    else:
        pretty["Recommendation path"] = "candidate backfill"
    pretty["Why this was recommended"] = pretty.apply(_friendly_reason, axis=1)
    pretty = pretty.rename(columns={"title": "Recommended product", "source_category": "Category"})
    print(title)
    display(
        pretty[
            [
                "Recommended product",
                "Category",
                "Price",
                "Average rating",
                "Recommendation path",
                "Confidence",
                "Why this was recommended",
            ]
        ].reset_index(drop=True)
    )


warm_user = str(splits.test_examples.sort_values("history_length", ascending=False).iloc[0]["user_id"])
sparse_user = str(splits.test_examples.sort_values("history_length", ascending=True).iloc[0]["user_id"])
cold_start_seed = prepared.item_features.sort_values(["average_rating", "log_positive_count"], ascending=[False, False]).head(3).copy()
cold_start_history = cold_start_seed["parent_asin"].tolist()

warm_history = get_user_order_history(prepared, splits, warm_user, split="test", limit=15)
sparse_history = get_user_order_history(prepared, splits, sparse_user, split="test", limit=15)

warm_df = recommend(prepared, splits, retrievers, ranker=ranker, user_id=warm_user, top_k=10)
sparse_df = recommend(prepared, splits, retrievers, ranker=ranker, user_id=sparse_user, top_k=10)
cold_df = recommend(prepared, splits, retrievers, ranker=ranker, history_items=cold_start_history, top_k=10)

_friendly_order_history_view(warm_history, f"Recent past orders before recommendation for user {warm_user}")
_friendly_recommendation_view(warm_df, f"Warm-user recommendations for user {warm_user}")
_friendly_order_history_view(sparse_history, f"Recent past orders before recommendation for user {sparse_user}")
_friendly_recommendation_view(sparse_df, f"Sparser-history recommendations for user {sparse_user}")

print("Cold-start seed products")
display(cold_start_seed[["parent_asin", "title", "source_category", "price", "average_rating"]].reset_index(drop=True))
_friendly_recommendation_view(cold_df, "Cold-start recommendations based on the seed products above")


Recent past orders before recommendation for user AG3NFZBQIXDZKJXRU5OI2KU3CQCQ


,Ordered at,Past order,Category,Review rating,Verified,Price,Catalog rating
0,2023-08-27,"CAMOTO H7 LED Headlight Bulbs,Super Bright 1:1 Mini Size H7 LED Bulb,Halogen Replacement Bulb 6500K Cool White,No Adapter Required,Easy ...",Automotive,5.0 / 5,No,$35.99,4.70 / 5
1,2023-08-27,"CAMOTO H7 LED Headlight Bulbs,Super Bright 1:1 Mini Size H7 LED Bulb,Halogen Replacement Bulb 6500K Cool White,No Adapter Required,Easy ...",Automotive,5.0 / 5,No,$35.99,4.70 / 5
2,2023-08-27,"CAMOTO H7 LED Headlight Bulbs,Super Bright 1:1 Mini Size H7 LED Bulb,Halogen Replacement Bulb 6500K Cool White,No Adapter Required,Easy ...",Automotive,5.0 / 5,No,$35.99,4.70 / 5
3,2023-08-27,"SUNLU 3D Printer Resin, Upgraded Standard Plus Fast Curing 3D Resin, 395 to 405nm UV Curing 3D Printing Photopolymer Resin, Higher Preci...",Industrial_and_Scientific,5.0 / 5,No,$19.99,4.80 / 5
4,2023-08-27,"SUNLU 3D Printer Resin, Upgraded Standard Plus Fast Curing 3D Resin, 395 to 405nm UV Curing 3D Printing Photopolymer Resin, Higher Preci...",Industrial_and_Scientific,5.0 / 5,No,$19.99,4.80 / 5
5,2023-08-27,"SUNLU 3D Printer Resin, Upgraded Standard Plus Fast Curing 3D Resin, 395 to 405nm UV Curing 3D Printing Photopolymer Resin, Higher Preci...",Industrial_and_Scientific,5.0 / 5,No,$19.99,4.80 / 5
6,2023-08-31,au-kee 9005 9006 LED Headlight Bulbs Upgraded 1:1 Mini Size 400% Brighter 6000K Cool White with Fan HB3 HB4 High Low Beam Halogen Replac...,Automotive,5.0 / 5,No,$23.26,4.00 / 5
7,2023-08-31,au-kee 9005 9006 LED Headlight Bulbs Upgraded 1:1 Mini Size 400% Brighter 6000K Cool White with Fan HB3 HB4 High Low Beam Halogen Replac...,Automotive,5.0 / 5,No,$23.26,4.00 / 5
8,2023-08-31,au-kee 9005 9006 LED Headlight Bulbs Upgraded 1:1 Mini Size 400% Brighter 6000K Cool White with Fan HB3 HB4 High Low Beam Halogen Replac...,Automotive,5.0 / 5,No,$23.26,4.00 / 5
9,2023-08-31,"SUNLU Upgraded S2 Filament Dryer Box with Fan, 360° Heating, Real-time Humidity Display, 3D Printer Filament Dehydrator for PLA, TPU, PE...",Industrial_and_Scientific,5.0 / 5,No,$69.99,4.60 / 5


Warm-user recommendations for user AG3NFZBQIXDZKJXRU5OI2KU3CQCQ


,Recommended product,Category,Price,Average rating,Recommendation path,Confidence,Why this was recommended
0,"SUNLU AntiString PLA Filament 1.75mm APLA 3D Printer Filament 1.75mm, 1kg Spool (2.2lbs), Dimensional Accuracy +/- 0.02mm, Neatly Wound ...",Industrial_and_Scientific,$15.99,4.40 / 5,"collaborative cooccurrence, latent CF",High,Similar shoppers showed related behavior; it is also strongly rated.
1,"taulman3D 1.75mm Nylon Bridge Filament Consumable, Polyamide (PA) 1lb Spool, Fits Nearly All FDM 3D Printers (Black)",Industrial_and_Scientific,$23.26,4.00 / 5,latent CF,High,Similar shoppers showed related behavior.
2,"3D Printer Filament PLA White-PLA Filament 1.75 mm SUNLU,Low Odor Dimensional Accuracy +/- 0.02 mm 3D Printing Filament,2.2 LBS (1KG) Sp...",Industrial_and_Scientific,$23.26,4.00 / 5,latent CF,Medium,Similar shoppers showed related behavior.
3,"3D Printer Filament PLA Black-PLA Filament 1.75 mm SUNLU,Low Odor Dimensional Accuracy +/- 0.02 mm 3D Printing Filament,2.2 LBS (1KG) Sp...",Industrial_and_Scientific,$23.26,4.20 / 5,latent CF,Medium,Similar shoppers showed related behavior.
4,"FRAM Fresh Breeze Cabin Air Filter Replacement for Car Passenger Compartment w/Arm and Hammer Baking Soda, Easy Install, CF9336 for Sele...",Automotive,$23.26,4.70 / 5,latent CF,Medium,Similar shoppers showed related behavior; it is also strongly rated.
5,"AUTOSAVER88 3"" Inlet/Outlet Universal Catalytic Converter w/Heat Shield (EPA Compliant)",Automotive,$48.35,4.50 / 5,latent CF,Exploratory,Similar shoppers showed related behavior; it is also strongly rated.
6,"PLA Glow in The Dark, Pink Rose 3D Printing Filament, 1.75 mm",Industrial_and_Scientific,$23.26,3.60 / 5,latent CF,Exploratory,Similar shoppers showed related behavior.
7,10 Set 4-Pin 5.08mm Pitch Male Female PCB Screw Terminal Block,Industrial_and_Scientific,$7.28,4.60 / 5,latent CF,Exploratory,Similar shoppers showed related behavior; it is also strongly rated.
8,"Vitodeco Genuine Leather Smart Key Fob Case Compatible with Lexus RX, Lexus ES, Lexus UX, Lexus NX, Lexus GX, Lexus LX 600 (4-Button, Bl...",Automotive,$14.99,4.30 / 5,latent CF,Exploratory,Similar shoppers showed related behavior.
9,HATCHBOX 3 Spool 3D Printer Filament Tabletop Wall Mount Rack,Industrial_and_Scientific,$23.26,3.20 / 5,latent CF,Exploratory,Similar shoppers showed related behavior.


Recent past orders before recommendation for user AHCBWXLXWYHXE3ORPANP5DCQME6A


...

,Recommended product,Category,Price,Average rating,Recommendation path,Confidence,Why this was recommended
0,"Hopkins Towing Solutions 40974 Multi-Tow 7 Blade and 4 Flat Connector (Packaging may vary),3.25 x 3.88 x 6.88,grey",Automotive,$27.84,4.70 / 5,latent CF,High,Similar shoppers showed related behavior; it is also strongly rated.
1,"Gas Can Spout Replacement, Gas Can Nozzle Kit with Screw Collar Caps, Gasket Stopper,and 2 Kinds Gas Can Vent Cap for Most Style Gas Can...",Automotive,$29.99,4.10 / 5,collaborative cooccurrence,High,Similar shoppers showed related behavior.
2,"Nilight ZH059 2PCS 4 Inch 18W Spot LED Light Mounting Bracket Horizontal Bar Tube Clamp with Off Road Wiring Harness-2 Leads, 2 Years Wa...",Automotive,$31.30,4.40 / 5,latent CF,Medium,Similar shoppers showed related behavior; it is also strongly rated.
3,GYEON Quartz LeatherShield 50ml - Advanced Sio2 Ceramic Coating for Leather - All Types of Natural Leather and Vegan Leather Alike - Doe...,Automotive,$59.99,4.60 / 5,"collaborative cooccurrence, latent CF",Medium,Similar shoppers showed related behavior; it is also strongly rated.
4,"Ready America 33111 Museum Gel, Clear",Industrial_and_Scientific,$12.93,4.40 / 5,latent CF,Medium,Similar shoppers showed related behavior; it is also strongly rated.
5,"BEAMTECH H4 LED Headlight Bulbs, 16000LM 70W 6500K Extremely Super Bright 9003 30mm Heatsink Base CSP Chips Conversion Kit,Xenon White S...",Automotive,$46.99,4.50 / 5,collaborative cooccurrence,Exploratory,Similar shoppers showed related behavior; it is also strongly rated.
6,"BYGD 150W Car Power Inverter, DC 12V to 110V AC Converter with Dual 2.4A USB Charging Ports Cigarette Lighter Socket Adapter",Automotive,$17.99,4.20 / 5,collaborative cooccurrence,Exploratory,Similar shoppers showed related behavior.
7,"VIHIMAI Tire Caps, Aluminum Alloy Valve Stem Cap, Decorative Accessory Universal Fit for Cars, SUV, Truck, Motorcycles, 4 Pack (Black)",Automotive,$6.99,4.70 / 5,"collaborative cooccurrence, latent CF",Exploratory,Similar shoppers showed related behavior; it is also strongly rated.
8,"LRTER 9005/HB3 LED Headlight Bulbs 110W High Power 20000LM Extremely Bright 6500K Cool White LED Headlights Conversion Kit Mini Size, Pa...",Automotive,$20.99,4.30 / 5,collaborative cooccurrence,Exploratory,Similar shoppers showed related behavior.
9,Personalized 4 Hole Chrome Metal Laser Engraved Standard Size (6”x12”) with Custom Text Design - Car License Plate Frame with Free caps ...,Automotive,$29.99,4.80 / 5,latent CF,Exploratory,Similar shoppers showed related behavior; it is also strongly rated.


Cold-start seed products


,parent_asin,title,source_category,price,average_rating
0,B08YX9M597,"No-Spill 1405 2-1/2-Gallon Poly Gas Can & STA-BIL Storage Fuel Stabilizer, 4 fl. oz. (22205), Red",Automotive,34.06,5.0
1,B07ZTQJ924,Motorcraft FD-4615 Fuel Filter,Automotive,94.78,5.0
2,B01I1OI3E6,"Drill America #4-40 1"" OD High Speed Steel Round Adjustable Die, DWT Series",Industrial_and_Scientific,14.24,5.0


Cold-start recommendations based on the seed products above


,Recommended product,Category,Price,Average rating,Recommendation path,Confidence,Why this was recommended
0,Motorcraft - Oil Filter (FL2051S),Automotive,$23.26,4.80 / 5,"collaborative cooccurrence, latent CF",High,Similar shoppers showed related behavior; it is also strongly rated.
1,"ELEGOO 3PCS 400 tie-Points breadboard, 4 Power Rails for Jumper Wire",Industrial_and_Scientific,$7.99,4.30 / 5,collaborative cooccurrence,High,Similar shoppers showed related behavior.
2,"Reese 21536 Drawbar 2 Inch Square and Ballmount Towing 2 Inch Starter Kit, Black",Automotive,$25.99,4.70 / 5,collaborative cooccurrence,Medium,Similar shoppers showed related behavior; it is also strongly rated.
3,HiLetgo 2000W PWM AC Motor Speed Control Module Dimmer Speed Regulator 50-220V Adjustable Voltage Regulator,Industrial_and_Scientific,$6.59,4.00 / 5,collaborative cooccurrence,Medium,Similar shoppers showed related behavior.
4,"230 Buna-N O-Ring, 70A Durometer, Black, 2-1/2"" ID, 2-3/4"" OD, 1/8"" Width (Pack of 10)",Industrial_and_Scientific,$9.38,4.60 / 5,collaborative cooccurrence,Medium,Similar shoppers showed related behavior; it is also strongly rated.
5,"Goplus 3/4'' Automatic Fuel Nozzle, Auto Shut Off Gas Pump Handle for Diesel Kerosene Biodiesel Fuel Refilling",Automotive,$34.99,4.20 / 5,"latent CF, content similarity",Exploratory,Similar shoppers showed related behavior; the product looks similar to past purchases.
6,Avenger MC0006 6-Inch Digital Caliper with Large Display,Industrial_and_Scientific,$23.26,3.40 / 5,collaborative cooccurrence,Exploratory,Similar shoppers showed related behavior.
7,"Superwinch 1145230 Terra 45 4500lbs/2046kg single line pull with hawse, handlebar mnt toggle, handheld remote, and synthetic rope",Automotive,$488.99,4.60 / 5,collaborative cooccurrence,Exploratory,Similar shoppers showed related behavior; it is also strongly rated.
8,"026 Buna O-Ring, 70A Durometer, Black, 1-1/4"" ID, 1-3/8"" OD, 1/16"" Width (Pack of 25)",Industrial_and_Scientific,$12.99,4.10 / 5,collaborative cooccurrence,Exploratory,Similar shoppers showed related behavior.
9,DEE ZEE DZ43206 Tailgate Assist Fits 19-Current Ford Ranger,Automotive,$29.16,4.70 / 5,"collaborative cooccurrence, latent CF",Exploratory,Similar shoppers showed related behavior; it is also strongly rated.
